In [5]:
import torch
import torchvision
import torch.nn as nn
import numpy as np
import tqdm

ModuleNotFoundError: No module named 'torchvision'

In [ ]:
from manify.curvature_estimation import delta_hyperbolicity

In [ ]:
class Flatten(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        B = x.shape[0]
        return x.view(B, -1)

def get_delta(loader, device):
    """
    computes delta value for image data by extracting features using VGG network;
    input -- data loader for images
    """
    vgg = torchvision.models.vgg16(pretrained=True)
    vgg_feats = vgg.features
    vgg_classifier = nn.Sequential(*list(vgg.classifier.children())[:-1])

    vgg_part = nn.Sequential(vgg_feats, Flatten(), vgg_classifier).to(device)
    vgg_part.eval()

    all_features = []
    for i, (batch, _) in enumerate(loader):
        with torch.no_grad():
            batch = batch.to(device)
            all_features.append(vgg_part(batch))

    all_features = torch.cat(all_features)
    idx = np.random.choice(len(all_features), 1500)
    all_features_small = all_features[idx]

    dists = torch.cdist(all_features_small, all_features_small, p=2)
    delta = delta_hyperbolicity(dists)
    diam = np.max(dists)
    return delta, diam